In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense, GlobalMaxPooling1D, GlobalAveragePooling1D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight


In [2]:

# Load data
df = pd.read_csv("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\nepal_data_vader_labeled2.csv")
X_text = df['lemmatized_text'].astype(str)
y_text = df['sentiment_textblob']


In [ ]:
pd.DataFrame()

In [3]:

# Label encode
le = LabelEncoder()
y = le.fit_transform(y_text)

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# Tokenize
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding='post')


In [5]:
# Build model
input_layer = Input(shape=(200,))
embedding = Embedding(input_dim=20000, output_dim=256)(input_layer)  # increased embedding
bilstm = Bidirectional(LSTM(512, return_sequences=True))(embedding) # increased LSTM units
avg_pool = GlobalAveragePooling1D()(bilstm)
max_pool = GlobalMaxPooling1D()(bilstm)
conc = Concatenate()([avg_pool, max_pool])
dense_feature = Dense(256, activation='relu', kernel_regularizer=l2(0.01), name='feature_layer')(conc)
dropout = Dropout(0.3)(dense_feature) # reduced dropout
output = Dense(3, activation='softmax')(dropout)

bilstm_classifier = Model(inputs=input_layer, outputs=output)
bilstm_classifier.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
bilstm_classifier.fit(
    X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_test_pad, y_test), verbose=1, class_weight=class_weight_dict, callbacks=[early_stopping]
)


Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 774s 8s/step - accuracy: 0.5418 - loss: 1.6472 - val_accuracy: 0.7576 - val_loss: 0.6837
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 645s 7s/step - accuracy: 0.8730 - loss: 0.4457 - val_accuracy: 0.8562 - val_loss: 0.4811
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 694s 7s/step - accuracy: 0.9542 - loss: 0.2009 - val_accuracy: 0.8704 - val_loss: 0.4162
Epoch 4/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 692s 7s/step - accuracy: 0.9778 - loss: 0.1088 - val_accuracy: 0.8801 - val_loss: 0.4593
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 687s 7s/step - accuracy: 0.9910 - loss: 0.0672 - val_accuracy: 0.8704 - val_loss: 0.4788
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 474s 5s/step - accuracy: 0.9882 - loss: 0.0696 - val_accuracy: 0.8788 - val_loss: 0.4612


In [6]:
# Feature extractor
feature_extractor = Model(
    inputs=bilstm_classifier.input,
    outputs=bilstm_classifier.get_layer('feature_layer').output
)

X_train_features = feature_extractor.predict(X_train_pad)
X_test_features = feature_extractor.predict(X_test_pad)

scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features)
X_test_features = scaler.transform(X_test_features)


194/194 ━━━━━━━━━━━━━━━━━━━━ 84s 428ms/step
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 411ms/step


In [7]:
# # Apply SMOTE
# smote = SMOTE(random_state=42)
# X_train_features_smote, y_train_smote = smote.fit_resample(X_train_features, y_train)

# params = {'C': [1, 10, 20], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto', 0.1, 1]}
# grid = GridSearchCV(
#     SVC(probability=True, class_weight=class_weight_dict),
#     params,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1
# )
# grid.fit(X_train_features_smote, y_train_smote)

# print(f"Best SVM Params: {grid.best_params_}")
# y_test_pred = grid.predict(X_test_features)
# acc = accuracy_score(y_test, y_test_pred)
# print(f"Accuracy with SVM: {acc:.4f}")
# print(classification_report(y_test, y_test_pred, target_names=le.classes_))

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV(
    SVC(probability=True, class_weight='balanced'),
    params,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid.fit(X_train_features, y_train)

y_test_pred = grid.predict(X_test_features)

print("Best Params:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy: 0.8878143133462283
              precision    recall  f1-score   support

    negative       0.81      0.84      0.82       381
     neutral       0.94      0.93      0.93       679
    positive       0.89      0.87      0.88       491

    accuracy                           0.89      1551
   macro avg       0.88      0.88      0.88      1551
weighted avg       0.89      0.89      0.89      1551



In [17]:
# --- Added SHAP Explainability ---
import shap

print("\nStarting SHAP explanation on a sample of around 200 rows of text...")
# Filter out texts with only 1 word, which crash the SHAP Text masker's hierarchical clustering
import re
valid_texts = []
for text in X_test_text.astype(str):
    if not pd.isna(text) and text.strip() != "":
        # Ensure the text has at least 2 words (SHAP PartitionExplainer needs >= 2 words to cluster)
        if len(re.findall(r"\w+", text)) >= 2:
            valid_texts.append(text)

shap_sample_texts = valid_texts[:200]

def predict_proba_text(texts):
    # Determine the shape/type of texts being passed by SHAP
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    
    seqs = tokenizer.texts_to_sequences(texts)
    pads = pad_sequences(seqs, maxlen=200, padding='post')
    # Use explicit batch_size to prevent out-of-memory errors
    features = feature_extractor.predict(pads, batch_size=32, verbose=0)
    features_scaled = scaler.transform(features)
    return grid.predict_proba(features_scaled)

# Create an explainer using a text masker
masker = shap.maskers.Text(r"\W+")
explainer = shap.Explainer(predict_proba_text, masker, output_names=list(le.classes_))

# Calculate SHAP values
# Adding max_evals to speed up execution and reduce memory overhead for 200 samples
try:
    shap_values = explainer(shap_sample_texts, max_evals=300)

    # Generate an HTML report and save it
    print("Generating SHAP explanation HTML...")
    shap_html = shap.plots.text(shap_values, display=False)
    with open("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\shap_explanation.html", "w", encoding="utf-8") as f:
        f.write(shap_html)
    print("SHAP explanation successfully saved to 'shap_explanation.html'")
except Exception as e:
    print(f"SHAP encountered an error: {e}")
    import traceback
    traceback.print_exc()



Starting SHAP explanation on a sample of around 200 rows of text...


  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   2%|▎         | 5/200 [00:00<?, ?it/s]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   4%|▍         | 9/200 [00:48<14:04,  4.42s/it]

...

  0%|          | 0/90 [00:00<?, ?it/s]

PartitionExplainer explainer:  99%|█████████▉| 198/200 [22:09<00:24, 12.02s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer: 201it [22:41,  6.94s/it]                         


Generating SHAP explanation HTML...
SHAP explanation successfully saved to 'shap_explanation.html'


In [18]:
print("\nStarting SHAP explanation on a sample of around 200 rows of text...")
# Filter out texts with only 1 word, which crash the SHAP Text masker's hierarchical clustering
import re
valid_texts = []
for text in X_test_text.astype(str):
    if not pd.isna(text) and text.strip() != "":
        # Ensure the text has at least 2 words (SHAP PartitionExplainer needs >= 2 words to cluster)
        if len(re.findall(r"\w+", text)) >= 2:
            valid_texts.append(text)

shap_sample_texts = valid_texts[:200]

def predict_proba_text(texts):
    # Determine the shape/type of texts being passed by SHAP
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    
    seqs = tokenizer.texts_to_sequences(texts)
    pads = pad_sequences(seqs, maxlen=200, padding='post')
    # Use explicit batch_size to prevent out-of-memory errors
    features = feature_extractor.predict(pads, batch_size=32, verbose=0)
    features_scaled = scaler.transform(features)
    return grid.predict_proba(features_scaled)

# Create an explainer using a text masker
masker = shap.maskers.Text(r"\W+")
explainer = shap.Explainer(predict_proba_text, masker, output_names=list(le.classes_))

# Calculate SHAP values
# Adding max_evals to speed up execution and reduce memory overhead for 200 samples
try:
    shap_values = explainer(shap_sample_texts, max_evals=300)

    # Generate an HTML report and save it
    print("Generating SHAP explanation HTML...")
    shap_html = shap.plots.text(shap_values, display=False)
    with open("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\shap_explanation.html", "w", encoding="utf-8") as f:
        f.write(shap_html)
    print("SHAP explanation successfully saved to 'shap_explanation.html'")
    
    # --- Extract top emotion words for all classes ---
    import collections
    print("\nExtracting top indicator words for EACH sentiment class...")
    
    for class_name in le.classes_:
        class_idx = list(le.classes_).index(class_name)
        word_shap_sums = collections.defaultdict(float)
        
        for i in range(len(shap_values)):
            for j, word in enumerate(shap_values.data[i]):
                clean_word = word.strip().lower()
                if clean_word:
                    val = shap_values.values[i][j][class_idx]
                    if val > 0: # Word contributed positively towards this exact class
                        word_shap_sums[clean_word] += val
                        
        top_words = sorted(word_shap_sums.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\n--- Top Words Driving '{class_name.upper()}' Prediction ---")
        for word, score in top_words[:20]:
            print(f" * {word} (SHAP importance: {score:.4f})")

except Exception as e:
    print(f"SHAP encountered an error: {e}")
    import traceback
    traceback.print_exc()



Starting SHAP explanation on a sample of around 200 rows of text...


PartitionExplainer explainer:   2%|▏         | 4/200 [00:00<?, ?it/s]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   3%|▎         | 6/200 [00:30<29:50,  9.23s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

...

  0%|          | 0/90 [00:00<?, ?it/s]

PartitionExplainer explainer:  99%|█████████▉| 198/200 [25:31<00:24, 12.06s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer: 201it [26:02,  7.93s/it]                         


Generating SHAP explanation HTML...
SHAP explanation successfully saved to 'shap_explanation.html'

Extracting top indicator words for EACH sentiment class...

--- Top Words Driving 'NEGATIVE' Prediction ---
 * corrupt (SHAP importance: 14.4584)
 * wrong (SHAP importance: 2.9403)
 * bad (SHAP importance: 2.1720)
 * destroy (SHAP importance: 2.1007)
 * military (SHAP importance: 1.8372)
 * violent (SHAP importance: 1.8164)
 * dead (SHAP importance: 1.4843)
 * fuck (SHAP importance: 1.4637)
 * useless (SHAP importance: 1.2638)
 * mean (SHAP importance: 1.2196)
 * crazy (SHAP importance: 1.2192)
 * serious (SHAP importance: 1.1714)
 * hard (SHAP importance: 1.1659)
 * foreign (SHAP importance: 1.1110)
 * difficult (SHAP importance: 1.0584)
 * brutal (SHAP importance: 0.9841)
 * stupid (SHAP importance: 0.9793)
 * dumb (SHAP importance: 0.9732)
 * evil (SHAP importance: 0.9287)
 * color (SHAP importance: 0.9100)

--- Top Words Driving 'NEUTRAL' Prediction ---
 * watch (SHAP importance: 0.5

In [21]:
with open(f"C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\top_words_{class_name}.txt", "w", encoding="utf-8") as f:
    for word, score in top_words:
        f.write(f"{word}\t{score:.6f}\n")

In [ ]:
file_path = r"C:\Users\madhu\OneDrive\Documents\New folder\hybrid model\top_words_positive.txt"

with open(file_path, "w", encoding="utf-8") as f:
    for word, score in top_words:
        f.write(f"{word}\t{score:.6f}\n")

In [22]:
try:
    shap_values = explainer(shap_sample_texts, max_evals=300)

    # Generate an HTML report and save it
    print("Generating SHAP explanation HTML...")
    shap_html = shap.plots.text(shap_values, display=False)
    with open("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\shap_explanation.html", "w", encoding="utf-8") as f:
        f.write(shap_html)
    print("SHAP explanation successfully saved to 'shap_explanation.html'")
    
    # --- Extract top emotion words for all classes ---
    import collections
    print("\nExtracting top indicator words for EACH sentiment class...")
    output_filepath = "C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\top_sentiment_words.txt"
    
    with open(output_filepath, "w", encoding="utf-8") as f_out:
        for class_name in le.classes_:
            class_idx = list(le.classes_).index(class_name)
            word_shap_sums = collections.defaultdict(float)
            
            for i in range(len(shap_values)):
                for j, word in enumerate(shap_values.data[i]):
                    clean_word = word.strip().lower()
                    if clean_word:
                        val = shap_values.values[i][j][class_idx]
                        if val > 0: # Word contributed positively towards this exact class
                            word_shap_sums[clean_word] += val
                            
            top_words = sorted(word_shap_sums.items(), key=lambda x: x[1], reverse=True)
            
            header = f"\n--- Top Words Driving '{class_name.upper()}' Prediction ---\n"
            print(header.strip())
            f_out.write(header)
            
            for word, score in top_words[:20]:
                line = f" * {word} (SHAP importance: {score:.4f})"
                print(line)
                f_out.write(line + "\n")
                
    print(f"\nSuccessfully saved top sentiment words to '{output_filepath}'")

except Exception as e:
    print(f"SHAP encountered an error: {e}")
    import traceback
    traceback.print_exc()


PartitionExplainer explainer:   2%|▎         | 5/200 [00:13<05:23,  1.66s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   3%|▎         | 6/200 [00:39<37:47, 11.69s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer:   4%|▍         | 9/200 [01:08<26:54,  8.45s/it]

...

  0%|          | 0/90 [00:00<?, ?it/s]

PartitionExplainer explainer:  99%|█████████▉| 198/200 [26:16<00:24, 12.07s/it]

  0%|          | 0/298 [00:00<?, ?it/s]

PartitionExplainer explainer: 201it [26:47,  8.12s/it]                         


Generating SHAP explanation HTML...
SHAP explanation successfully saved to 'shap_explanation.html'

Extracting top indicator words for EACH sentiment class...
--- Top Words Driving 'NEGATIVE' Prediction ---
 * corrupt (SHAP importance: 14.4584)
 * wrong (SHAP importance: 2.9403)
 * bad (SHAP importance: 2.1720)
 * destroy (SHAP importance: 2.1007)
 * military (SHAP importance: 1.8372)
 * violent (SHAP importance: 1.8164)
 * dead (SHAP importance: 1.4843)
 * fuck (SHAP importance: 1.4637)
 * useless (SHAP importance: 1.2638)
 * mean (SHAP importance: 1.2196)
 * crazy (SHAP importance: 1.2192)
 * serious (SHAP importance: 1.1714)
 * hard (SHAP importance: 1.1659)
 * foreign (SHAP importance: 1.1110)
 * difficult (SHAP importance: 1.0584)
 * brutal (SHAP importance: 0.9841)
 * stupid (SHAP importance: 0.9793)
 * dumb (SHAP importance: 0.9732)
 * evil (SHAP importance: 0.9287)
 * color (SHAP importance: 0.9100)
--- Top Words Driving 'NEUTRAL' Prediction ---
 * watch (SHAP importance: 0.540

In [25]:
# --- Determine the Overall Dominant Sentiment (Independently) ---
try:
    if 'class_total_impacts' in locals() and class_total_impacts:
        dominant_sentiment = max(class_total_impacts, key=class_total_impacts.get)
        dominant_msg = f"\n=== OVERALL DOMINANT SENTIMENT: '{dominant_sentiment.upper()}' ==="
        dominant_msg += f"\n(Based on total cumulative SHAP importance: {class_total_impacts[dominant_sentiment]:.4f})\n"
        
        print(dominant_msg)
        with open("C:\\Users\\madhu\\OneDrive\\Documents\\New folder\\hybrid model\\top_sentiment_words.txt", "a", encoding="utf-8") as f_out_append:
            f_out_append.write(dominant_msg)
except NameError:
    pass



In [26]:
import collections
# 1. Compute total SHAP impacts for each class from existing shap_values
class_total_impacts = {}
for class_name in le.classes_:
    class_idx = list(le.classes_).index(class_name)
    total_class_impact = 0.0
    
    # Iterate through each sample and aggregate exact positive impacts
    for i in range(len(shap_values)):
        for j, word in enumerate(shap_values.data[i]):
            clean_word = word.strip().lower()
            if clean_word:
                val = shap_values.values[i][j][class_idx]
                
                # Only sum the positive impacts driving towards this specific class
                if val > 0: 
                    total_class_impact += val
                    
    class_total_impacts[class_name] = total_class_impact
# 2. Determine and print the dominant sentiment overall
if class_total_impacts:
    dominant_sentiment = max(class_total_impacts, key=class_total_impacts.get)
    
    print(f"=== OVERALL DOMINANT SENTIMENT: '{dominant_sentiment.upper()}' ===\n")
    
    # Print the breakdown for visibility 
    print("Impact Breakdown by Sentiment:")
    for sentiment, impact in class_total_impacts.items():
        if sentiment == dominant_sentiment:
            print(f" -> {sentiment.upper()}: {impact:.4f} (DOMINANT)")
        else:
            print(f"  - {sentiment.lower()}: {impact:.4f}")
else:
    print("No SHAP values were found to process! Run the explainer first.")

=== OVERALL DOMINANT SENTIMENT: 'POSITIVE' ===

Impact Breakdown by Sentiment:
  - negative: 67.6779
  - neutral: 6.8342
 -> POSITIVE: 79.8321 (DOMINANT)
